# Coffee Quality Analysis: Sensory & Physical Profiles
**Author:** Janavika Shanmuga  
**Date:** July 17, 2026  
**Project Description:** An exploratory data analysis (EDA) of the Coffee Quality Institute (CQI) Arabica dataset, examining relationships between altitude, processing methods, physical defects, and sensory attributes.

In [1]:
import pandas as pd

# Used the GitHub URL for the CQI Arabica dataset
url = "https://raw.githubusercontent.com/fatih-boyar/coffee-quality-data-CQI/main/df_arabica_clean.csv"

# Load the dataset
df = pd.read_csv(url)

# Print confirmation and sample data
print(f"Dataset successfully loaded with {df.shape[0]} rows and {df.shape[1]} columns!")
print(df[['Country of Origin', 'Total Cup Points', 'Altitude', 'Processing Method']].head())

Dataset successfully loaded with 207 rows and 41 columns!
  Country of Origin  Total Cup Points   Altitude        Processing Method
0          Colombia             89.33  1700-1930  Double Anaerobic Washed
1            Taiwan             87.58       1200             Washed / Wet
2              Laos             87.42       1300              Semi Washed
3        Costa Rica             87.17       1900             Washed / Wet
4          Colombia             87.08  1850-2100             Honey,Mossto


In [2]:
import plotly.io as pio


coffee_template = pio.templates["plotly_white"]
coffee_template.layout.update(
    font=dict(family="Arial, sans-serif", size=12, color="#2c2c2c"),
    title=dict(font=dict(size=16, color="#1a1a1a", weight="bold"), x=0.05),
    margin=dict(t=80, b=50, l=60, r=30),
    xaxis=dict(showgrid=False, linecolor="#cccccc", ticks="outside"),
    yaxis=dict(showgrid=True, gridcolor="#f0f0f0", linecolor="#cccccc", ticks="outside")
)
pio.templates.default = coffee_template

In [ ]:
# 1. Quality vs. Altitude
import pandas as pd
import plotly.express as px
import numpy as np

#Converting the Altitude to numeric first
df_clean = df.copy()
df_clean['Altitude'] = pd.to_numeric(df_clean['Altitude'], errors='coerce')

#Filter out missing values and extreme altitude outliers
df_clean = df_clean[df_clean['Altitude'].notna() & (df_clean['Altitude'] < 3000)]
df_clean = df_clean.dropna(subset=['Total Cup Points'])

#Identify the peak performer to highlight it
highest_score = df_clean['Total Cup Points'].max()
df_clean['Highlight'] = np.where(df_clean['Total Cup Points'] == highest_score, 'Highest Quality', 'Other Coffees')

#Plotly scatter plot
fig1 = px.scatter(
    df_clean,
    x='Altitude',
    y='Total Cup Points',
    color='Highlight',
    color_discrete_map={'Other Coffees': "#455dbb", 'Highest Quality': '#D4AF37'},
    opacity=0.8,
    trendline="ols",
    trendline_color_override="#D32424",
    hover_data=['Country of Origin', 'Variety'],
    labels={'Altitude': 'Altitude (meters above sea level)', 'Total Cup Points': 'Overall Cup Score (100pt scale)'}
)

fig1.update_layout(
    title="<b>Altitude Premium:</b> High Altitude Farms Yield the Highest-Scoring Beans<br><span style='font-size:12px; color:#555;'>Sensory scores consolidate and peak as elevation approaches 2,000 meters.</span>",
    showlegend=False,
    width=600,
    height=600
)

# Show the plot
fig1.show()

In [ ]:
# 2. The Defect Profile (African vs. Central American Coffee)
import pandas as pd
import plotly.express as px

# Classifying countries into regions
def classify_region(country):
    african = ['Ethiopia', 'Kenya', 'Uganda', 'Tanzania', 'Rwanda', 'Burundi']
    central_am = ['Guatemala', 'Honduras', 'Nicaragua', 'Costa Rica', 'El Salvador', 'Panama']
    if country in african:
        return 'East Africa'
    elif country in central_am:
        return 'Central America'
    return 'Other'

# Applying classification to our cleaned dataframe
df_clean['Region'] = df_clean['Country of Origin'].apply(classify_region)
df_regional = df_clean[df_clean['Region'] != 'Other'].copy()

#Calculate the average defects by type for each region
defect_summary = df_regional.groupby('Region')[['Category One Defects', 'Category Two Defects']].mean().reset_index()

# Melt the dataframe so it's in a long format
defect_melted = defect_summary.melt(
    id_vars='Region', 
    var_name='Defect Type', 
    value_name='Average Defects per Batch'
)

# Clean up the labels for the chart
defect_melted['Defect Type'] = defect_melted['Defect Type'].replace({
    'Category One Defects': 'Primary (Category 1) Defects',
    'Category Two Defects': 'Secondary (Category 2) Defects'
})

#Create a clean, grouped bar chart
fig2 = px.bar(
    defect_melted,
    x='Defect Type',
    y='Average Defects per Batch',
    color='Region',
    barmode='group',
    color_discrete_map={'East Africa': "#6f3c0a", 'Central America': "#c29667"},
    labels={'Average Defects per Batch': 'Average Count per Batch (350g)'}
)

fig2.update_layout(
    title="<b>Regional Cleanliness:</b> Central American Beans Show Fewer Primary Defects<br><span style='font-size:12px; color:#555;'>Primary defects are rare across both regions, while secondary defects are more common.</span>",
    legend=dict(title="Production Region", yanchor="top", y=0.95, xanchor="right", x=0.95),
    width=650,
    height=600
)

fig2.show()

In [ ]:
# 3. Moisture Content of Raw Beans Across Top 5 Origins
import pandas as pd
import plotly.express as px

# Finding the top 5 countries by number of coffee entries
top_countries = df_clean['Country of Origin'].value_counts().head(5).index.tolist()
df_top5 = df_clean[df_clean['Country of Origin'].isin(top_countries)].copy()

# Calculate the average moisture percentage for these countries
moisture_summary = df_top5.groupby('Country of Origin')['Moisture Percentage'].mean().reset_index()
moisture_summary = moisture_summary.sort_values(by='Moisture Percentage', ascending=True)

# Highlight the country with the highest average moisture
highest_moisture_country = moisture_summary.loc[moisture_summary['Moisture Percentage'].idxmax(), 'Country of Origin']
moisture_summary['Color_Group'] = moisture_summary['Country of Origin'].apply(
    lambda x: 'Highest Moisture' if x == highest_moisture_country else 'Other'
)

#Create a clean, horizontal bar chart
fig3 = px.bar(
    moisture_summary,
    x='Moisture Percentage',
    y='Country of Origin',
    orientation='h',
    color='Color_Group',
    color_discrete_map={'Other': "#d4bdb1", 'Highest Moisture': '#8c6239'},
    labels={'Moisture Percentage': 'Average Moisture Content (%)', 'Country of Origin': 'Country of Origin'}
)

fig3.update_layout(
    title=f"<b>Moisture Consistency:</b> {highest_moisture_country} Shows Highest Bean Moisture Among Top Producers<br><span style='font-size:12px; color:#555;'>All top producers manage to maintain moisture within the ideal green coffee window of 9-12%.</span>",
    showlegend=False,
    yaxis=dict(autorange="reversed"),
    width=750,
    height=600
)

fig3.show()

In [ ]:
# 4. Processing Methods vs. Flavor Profiles
import pandas as pd
import plotly.express as px

# Filter for the two most prominent processing methods
df_proc = df_clean[df_clean['Processing Method'].isin(['Washed / Wet', 'Natural / Dry'])].copy()

#Create an elegant box plot to compare the sensory Flavor scores
fig4 = px.box(
    df_proc,
    x='Processing Method',
    y='Flavor',
    color='Processing Method',
    color_discrete_map={'Washed / Wet': "#DD3176", 'Natural / Dry': "#48a533"},
    points="outliers",
    labels={'Flavor': 'Flavor Score (0-10 scale)', 'Processing Method': 'Processing Method'}
)

fig4.update_layout(
    title="<b>Flavor Characteristics:</b> Natural Processing Unlocks Higher Flavor Peaks<br><span style='font-size:12px; color:#555;'>While median scores remain highly competitive, Natural/Dry processing yields a higher density of elite flavor ratings.</span>",
    showlegend=False,
    width=720,
    height=600
)

fig4.show()

In [ ]:
# 5. Bean Variety Quality (Balance vs. Acidity)
import pandas as pd
import plotly.express as px
import numpy as np

#Filter for the most common varieties to keep the chart clean
target_varieties = ['Bourbon', 'Caturra', 'Typica', 'Geisha']
df_var = df_clean[df_clean['Variety'].isin(target_varieties)].copy()

# Drop rows with missing values in our key attributes
df_var = df_var.dropna(subset=['Acidity', 'Balance'])

# Create a highlight column specifically for Geisha
df_var['Highlight'] = np.where(df_var['Variety'] == 'Geisha', 'Geisha (Premium)', 'Standard Varieties')

#Build the scatter plot
fig5 = px.scatter(
    df_var,
    x='Acidity',
    y='Balance',
    color='Highlight',
    color_discrete_map={'Standard Varieties': "#0abd4c", 'Geisha (Premium)': '#D4AF37'}, # Neutral gray vs Gold
    opacity=0.8,
    hover_data=['Country of Origin', 'Variety'],
    labels={'Acidity': 'Acidity Score (0-10)', 'Balance': 'Balance Score (0-10)'}
)

fig5.update_layout(
    title="<b>The Geisha Distinction:</b> Premium Geisha Beans Dominate High Balance-Acidity Scales<br><span style='font-size:12px; color:#555;'>While standard varieties clump below 8.0, Geisha beans consistently score near-perfect marks in both dimensions.</span>",
    legend=dict(title="Variety Class", yanchor="bottom", y=0.05, xanchor="right", x=0.95),
    width=800,
    height=600
)

fig5.show()

In [8]:
# 6. Raw Bean Color vs. Secondary Defects
import pandas as pd
import plotly.express as px

color_col = next((col for col in df_clean.columns if col.lower() in ['color', 'raw_bean_color']), None)
defect_col = next((col for col in df_clean.columns if col.lower().replace(' ', '_').replace('.', '_') in ['category_two_defects', 'category_2_defects', 'secondary_defects']), None)

if color_col and defect_col:
    #Normalize casing to prevent mismatches
    df_clean[color_col] = df_clean[color_col].astype(str).str.title().str.strip()
    
    # Filter for target colors
    main_colors = ['Green', 'Blue-Green', 'Yellow-Green']
    df_color = df_clean[df_clean[color_col].isin(main_colors)].copy()

    #Build the Box Plot
    fig6 = px.box(
        df_color,
        x=color_col,
        y=defect_col,
        color=color_col,
        color_discrete_map={
            'Green': "#18543e",       
            'Blue-Green': "#f17c07",
            'Yellow-Green': "#aee440"
        },
        points="outliers",
        labels={defect_col: 'Secondary Defects per Batch', color_col: 'Raw Bean Color'}
    )

    # Style the layout
    fig6.update_layout(
        title="<b>Color Quality:</b> Blue-Green Raw Beans Maintain the Cleanest Defect Profiles<br><span style='font-size:12px; color:#555;'>Muted yellow-green beans show a wider distribution and higher median counts of secondary defects.</span>",
        showlegend=False,
        width=650,
        height=600,
        plot_bgcolor='white',
        xaxis=dict(
            type='category', 
            categoryorder='array',
            categoryarray=main_colors
        ),
        yaxis=dict(
            gridcolor='#F0F0F0'
        )
    )

    fig6.show()

In [9]:
#7. The Acidity vs. Body Trade-off
import pandas as pd
import plotly.express as px

# Filter for the two primary processing methods
df_tradeoff = df_clean[df_clean['Processing Method'].isin(['Washed / Wet', 'Natural / Dry'])].copy()
df_tradeoff = df_tradeoff.dropna(subset=['Acidity', 'Body'])

#Build the scatter plot
fig7 = px.scatter(
    df_tradeoff,
    x='Acidity',
    y='Body',
    color='Processing Method',
    color_discrete_map={
        'Washed / Wet': "#eb7c06",   
        'Natural / Dry': "#694828"
    },
    opacity=0.7,
    trendline="ols",
    labels={'Acidity': 'Acidity Score (0-10)', 'Body': 'Body Score (0-10)'}
)

fig7.update_layout(
    title="<b>The Sensory Profile:</b> Processing Methods Segregate Acidity and Body Dynamics<br><span style='font-size:12px; color:#555;'>Natural/Dry methods pull profiles toward heavy body, while Washed/Wet methods lean cleanly toward high acidity.</span>",
    legend=dict(title="Processing Method", yanchor="top", y=0.95, xanchor="left", x=0.05),
    width=700,
    height=600
)

fig7.show()

In [10]:
#8. The Ultimate Predictor of Coffee Quality
import pandas as pd
import plotly.express as px

#Calculate correlation of sensory attributes with Total Cup Points
sensory_cols = ['Aroma', 'Flavor', 'Acidity', 'Balance', 'Body']
correlations = df_clean[sensory_cols].corrwith(df_clean['Total Cup Points']).reset_index()
correlations.columns = ['Sensory Attribute', 'Correlation']

# Sort from lowest to highest correlation for a clean horizontal bar flow
correlations = correlations.sort_values(by='Correlation', ascending=True)

# Highlight the absolute strongest predictor
strongest_predictor = correlations.loc[correlations['Correlation'].idxmax(), 'Sensory Attribute']
correlations['Highlight'] = correlations['Sensory Attribute'].apply(
    lambda x: 'Strongest Predictor' if x == strongest_predictor else 'Other Attributes'
)

#Build the clean horizontal bar chart
fig8 = px.bar(
    correlations,
    x='Correlation',
    y='Sensory Attribute',
    orientation='h',
    color='Highlight',
    color_discrete_map={'Other Attributes': '#d3d3d3', 'Strongest Predictor': '#D4AF37'},
    labels={'Correlation': 'Correlation Coefficient (r) with Total Cup Points', 'Sensory Attribute': 'Sensory Attribute'}
)

fig8.update_layout(
    title=f"<b>The Quality Blueprint:</b> '{strongest_predictor}' is the Strongest Predictor of Overall Cup Score<br><span style='font-size:12px; color:#555;'>While all attributes correlate positively, professional cuppers' evaluation of balance and flavor heavily dictates final scores.</span>",
    showlegend=False,
    xaxis=dict(range=[0, 1]),
    width=750, 
    height=600
)

fig8.show()

In [11]:
# 9. Flavor Profiles Across Continents
import pandas as pd
import plotly.graph_objects as go

#Broadly categorize countries into global macro-regions
def classify_continent(country):
    americas = ['Guatemala', 'Honduras', 'Nicaragua', 'Costa Rica', 'El Salvador', 'Panama', 'Mexico', 'Colombia', 'Brazil']
    africa = ['Ethiopia', 'Kenya', 'Uganda', 'Tanzania', 'Rwanda', 'Burundi']
    asia = ['Taiwan', 'Thailand', 'Indonesia', 'Vietnam', 'Myanmar']
    
    if country in americas:
        return 'Americas'
    elif country in africa:
        return 'Africa'
    elif country in asia:
        return 'Asia'
    return 'Other'

df_clean['Continent'] = df_clean['Country of Origin'].apply(classify_continent)
df_cont = df_clean[df_clean['Continent'] != 'Other'].copy()

# Calculate mean scores for core sensory dimensions
sensory_features = ['Aroma', 'Flavor', 'Acidity', 'Body', 'Balance']
continent_summary = df_cont.groupby('Continent')[sensory_features].mean().reset_index()

#Build the Radar Chart using Graph Objects (go) for precision control
fig9 = go.Figure()

# Define a cohesive warm palette for the three continents
colors = {'Africa': '#8c6239', 'Americas': '#c49a6c', 'Asia': '#555555'}

for index, row in continent_summary.iterrows():
    continent = row['Continent']
    values = row[sensory_features].tolist()
    values.append(values[0])
    
    fig9.add_trace(go.Scatterpolar(
        r=values,
        theta=sensory_features + [sensory_features[0]],
        fill='toself',
        name=continent,
        line=dict(color=colors[continent], width=2),
        fillcolor=colors[continent],
        opacity=0.15
    ))

#Apply elegant styling
fig9.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[7.2, 8.0],
            gridcolor="#f0f0f0"
        ),
        angularaxis=dict(
            gridcolor="#f0f0f0"
        )
    ),
    title="<b>Sensory Fingerprints:</b> African Beans Lead in Aroma and Acidity Metrics<br><span style='font-size:12px; color:#555;'>Asian and American beans offer balanced, body-forward characteristics with tighter sensory profiles.</span>",
    legend=dict(title="Production Origin", yanchor="top", y=0.95, xanchor="right", x=0.95),
    showlegend=True,
    width=700,
    height=600
)

fig9.show()

In [12]:
# 10. Processing Methods within the Elite Tier (Top 10%)
import pandas as pd
import plotly.express as px

#Determine the 90th percentile threshold for Total Cup Points
score_threshold = df_clean['Total Cup Points'].quantile(0.90)

# Filter for the "Elite" coffee beans
df_elite = df_clean[df_clean['Total Cup Points'] >= score_threshold].copy()

# Calculate the percentage of each processing method within the elite tier
elite_processing = df_elite['Processing Method'].value_counts(normalize=True).reset_index()
elite_processing.columns = ['Processing Method', 'Proportion']
elite_processing['Percentage'] = elite_processing['Proportion'] * 100

# Sort for a clean visual flow
elite_processing = elite_processing.sort_values(by='Percentage', ascending=True)

# Highlight the dominant processing method
dominant_method = elite_processing.loc[elite_processing['Percentage'].idxmax(), 'Processing Method']
elite_processing['Highlight'] = elite_processing['Processing Method'].apply(
    lambda x: 'Dominant Method' if x == dominant_method else 'Other Methods'
)

#Build the horizontal bar chart
fig10 = px.bar(
    elite_processing,
    x='Percentage',
    y='Processing Method',
    orientation='h',
    color='Highlight',
    color_discrete_map={'Other Methods': "#e8cfbc", 'Dominant Method': '#8c6239'},
    labels={'Percentage': 'Proportion of Elite Coffees (%)', 'Processing Method': 'Processing Method'}
)

fig10.update_layout(
    title=f"<b>The Elite Tier:</b> '{dominant_method}' Processing Dominates the Top 10% of Coffees<br><span style='font-size:12px; color:#555;'>Washed processing accounts for the vast majority of coffees scoring above the 90th percentile ({score_threshold:.2f} points).</span>",
    showlegend=False,
    xaxis=dict(ticksuffix="%"),
    width=750,
    height=600
)

fig10.show()

In [13]:
#11. Altitude Distributions Across Top 5 Producing Countries
import pandas as pd
import plotly.express as px

#Identify the top 5 countries by volume
top_countries = df_clean['Country of Origin'].value_counts().head(5).index.tolist()
df_top5_alt = df_clean[df_clean['Country of Origin'].isin(top_countries)].copy()

# Sort countries by their median altitude to make the chart easy to read
median_altitudes = df_top5_alt.groupby('Country of Origin')['Altitude'].median().sort_values(ascending=False).index.tolist()

#Build the multi-group box plot
fig11 = px.box(
    df_top5_alt,
    x='Country of Origin',
    y='Altitude',
    color='Country of Origin',
    category_orders={'Country of Origin': median_altitudes},
    color_discrete_sequence=["#643b12", "#3a2815", '#c49a6c', '#d9b48f', "#620f0f"],
    points="outliers",
    labels={'Altitude': 'Altitude (meters above sea level)', 'Country of Origin': 'Country of Origin'}
)

fig11.update_layout(
    title="<b>Geographic Sweet Spots:</b> Central American Farms Span Wider Elevation Ranges than East African Peers<br><span style='font-size:12px; color:#555;'>Countries are sorted by median altitude, revealing highly distinct microclimate bands for each origin.</span>",
    showlegend=False,
    width=900,
    height=600
)

fig11.show()

In [14]:
# 12. Flavor vs. Acidity Map by Raw Bean Color
import pandas as pd
import plotly.express as px

# Clean up whitespace and title-case the colors
df_clean['Color'] = df_clean['Color'].astype(str).str.strip().str.title()

# Automatically match whichever colors exist in dataset
preferred_colors = ['Dark Brown', 'Light Brown', 'Yellow-White']
available_colors = [c for c in preferred_colors if c in df_clean['Color'].unique()]

if not available_colors:
    main_colors = ['Green', 'Blue-Green', 'Yellow-Green']
    color_map = {
        'Green': "#18543e",       
        'Blue-Green': "#f17c07",
        'Yellow-Green': "#aee440"
    }
    subtitle_text = "Blue-Green and Green beans consistently trend higher, showcasing superior flavor-to-acidity ratios."
else:
    main_colors = available_colors
    color_map = {
        'Dark Brown': "#361e06",
        'Light Brown': "#935f27",     
        'Yellow-White': '#d3d3d3' 
    }
    subtitle_text = "Brown and Yellow-White beans consistently maintain tight linear trends, proving that flavor scales reliably with acidity."

# Filter dataset and build the plot
df_sensory_map = df_clean[df_clean['Color'].isin(main_colors)].copy().dropna(subset=['Flavor', 'Acidity'])

fig12 = px.scatter(
    df_sensory_map,
    x='Flavor',
    y='Acidity',
    color='Color',
    color_discrete_map=color_map,
    opacity=0.75,
    trendline="ols",
    labels={'Flavor': 'Flavor Score (0-10)', 'Acidity': 'Acidity Score (0-10)'}
)

fig12.update_layout(
    title=f"<b>Sensory Alignment:</b> Strong Flavor-Acidity Harmony Across All Bean Colors<br><span style='font-size:12px; color:#555;'>{subtitle_text}</span>",
    legend=dict(title="Raw Bean Color", yanchor="bottom", y=0.05, xanchor="right", x=0.95),
    width=700,
    height=600,
    plot_bgcolor='white'
)

fig12.show()

## Summary of Insights
* **Altitude Influence:** Higher altitude (approaching 2,000m) correlates with superior sensory scores.
* **Processing Impact:** Natural processing yields elite flavor profiles, while Washed processing ensures cleaner acidity.
* **Quality Predictors:** 'Flavor' is the most significant statistical driver of a coffee's total cup score, outweighing body or aroma.